# Préparons nos trois extraits

Conserver les mesures et les annotations sans confondre leurs axes.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#tenseur-forme-type-device).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Dans une petite sonothèque, nous voulons séparer voix et percussions. Commençons avec trois extraits, assez peu pour suivre chaque valeur. Une percussion sera d’abord mal classée : nous chercherons ensuite à réparer la règle.

Le trajet tient en une phrase : donner des mesures au modèle, comparer sa réponse aux annotations, puis ajuster ses paramètres. PyTorch fournit les tableaux, les couches et les outils de dérivation nécessaires. Nous les prendrons au moment où ils servent.

| Extrait | RMS choisie | Facteur de crête choisi | Annotation |
| --- | --- | --- | --- |
| Voix A | 0,10 | 1,5 | 0 : voix |
| Percussion B | 0,20 | 4,5 | 1 : percussion |
| Voix C | 0,15 | 2,5 | 0 : voix |

Ce sont des valeurs choisies pour le cours. Elles ne proviennent pas des lecteurs audio du site. La RMS résume le niveau ; le facteur de crête compare le pic à cette RMS.

Chaque ligne doit rester associée à sa réponse attendue. Nous plaçons les mesures dans X et les annotations dans cibles. Un tenseur est ici un tableau numérique auquel PyTorch associe notamment une forme et un type.

dtype choisit le stockage. float32 convient aux mesures décimales ; long est un autre nom de int64, l’entier signé sur 64 bits. Nos classes sont des indices entiers. La perte de classification expliquera pourquoi ce type est attendu ici. Pour suivre ses variations avec autograd, un tenseur doit être flottant ou complexe : les indices entiers ne sont pas des paramètres à dériver.

Si les indices sont déjà dans un tenseur nommé labels, labels.long() renvoie ces indices au type int64. Nous l’utiliserions pour adapter ces annotations au contrat de la perte.

X.shape vaut (3,2) : trois extraits, puis deux mesures. cibles.shape vaut (3,) : une annotation par extrait. La virgule de (3,) appartient à l’écriture d’un tuple avec une seule taille.

Essayons maintenant de retrouver le facteur de crête de A. Les indices commencent à zéro : le premier indice choisit l’extrait, le second la mesure.

| Objet | Contenu | Forme |
| --- | --- | --- |
| X | Les trois lignes de mesures | (3,2) |
| premiere_ligne | [0,10 ; 1,5] | (2,) |
| facteur_A | 1,5 | () |

Sélectionner une case laisse un tenseur scalaire, sans axe. Cela ne modifie pas le tableau X.

Pour afficher seulement son nombre, facteur_A.item() renvoie ici le nombre Python 1,5. X.dtype renseigne le type ; X.device renseigne le placement. Le programme travaille sur cpu, le processeur principal de l’ordinateur.

torch.isfinite(X) teste chaque case : elle doit être finie, sans infini ni NaN, une valeur numérique indéfinie. all() réduit ces booléens à un seul test ; item() en extrait un booléen Python. Ce contrôle lit les données, sans les réparer. Un affichage comme 0.10000000149 vient aussi parfois de l’approximation float32 de 0,1.

Les données sont prêtes, mais elles ne portent encore aucune prédiction. Donnons maintenant deux scores à chaque extrait et voyons comment une réponse du modèle devient une décision.

## Démonstration 1 · Exécuter l’expérience entière

Retrouver les préparations et suivre les valeurs dans l’ordre du programme.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [3]:
import torch


### Calculer l’état suivant

Les deux tableaux partagent le même axe des extraits. Les annotations ne sont pas ajoutées aux mesures fournies au modèle.

In [4]:
X = torch.tensor([[0.1, 1.5], [0.2, 4.5], [0.15, 2.5]], dtype=torch.float32)
cibles = torch.tensor([0, 1, 0], dtype=torch.long)


In [5]:
observer('X', X)
observer('cibles', cibles)

X forme (3, 2) type torch.float32 requires_grad False
  valeurs : [[0.10000000149011612, 1.5], [0.20000000298023224, 4.5], [0.15000000596046448, 2.5]]
cibles forme (3,) type torch.int64 requires_grad False
  valeurs : [0, 1, 0]


### Calculer l’état suivant

Le premier indice prend une ligne ; les deux indices prennent une case de cette ligne. X garde ses valeurs et sa forme.

In [6]:
premiere_ligne = X[0]
facteur_A = X[0, 1]


In [7]:
observer('premiere_ligne', premiere_ligne)
observer('facteur_A', facteur_A)

premiere_ligne forme (2,) type torch.float32 requires_grad False
  valeurs : [0.10000000149011612, 1.5]
facteur_A forme () type torch.float32 requires_grad False
  valeurs : 1.5


### Lire les sorties

shape, dtype et device décrivent l’objet. item extrait le seul nombre du tenseur scalaire pour l’afficher.

In [8]:
print("formes :", tuple(X.shape), tuple(cibles.shape))
print("selection :", tuple(premiere_ligne.shape), tuple(facteur_A.shape))
print("facteur A :", facteur_A.item())
print("types :", X.dtype, cibles.dtype, "device :", X.device)


formes : (3, 2) (3,)
selection : (2,) ()
facteur A : 1.5
types : torch.float32 torch.int64 device : cpu


### Calculer l’état suivant

isfinite pose une question par case. all les rassemble ; item sort le booléen scalaire dans Python.

In [9]:
finies = torch.isfinite(X)
toutes_finies = finies.all()
controle = toutes_finies.item()


In [10]:
observer('finies', finies)
observer('toutes_finies', toutes_finies)
observer('controle', controle)

finies forme (3, 2) type torch.bool requires_grad False
  valeurs : [[True, True], [True, True], [True, True]]
toutes_finies forme () type torch.bool requires_grad False
  valeurs : True
controle = True


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [11]:
print("valeurs finies :", controle)

valeurs finies : True


In [12]:
assert bool(tuple(X.shape) == (3, 2) and tuple(cibles.shape) == (3,)), 'tenseur-forme-type-device/main: tuple(X.shape) == (3, 2) and tuple(cibles.shape) == (3,)'
assert bool(X.dtype == torch.float32 and cibles.dtype == torch.int64), 'tenseur-forme-type-device/main: X.dtype == torch.float32 and cibles.dtype == torch.int64'
assert bool(tuple(premiere_ligne.shape) == (2,) and tuple(facteur_A.shape) == () and facteur_A.item() == 1.5), 'tenseur-forme-type-device/main: tuple(premiere_ligne.shape) == (2,) and tuple(facteur_A.shape) == () and facteur_A.item() == 1.5'
assert bool(controle is True and finies.shape == X.shape), 'tenseur-forme-type-device/main: controle is True and finies.shape == X.shape'
print("Contrôles numériques de la démonstration réussis.")

Contrôles numériques de la démonstration réussis.


## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Choisissez le dtype des indices de classes [0,1,0] pour CrossEntropyLoss dans son mode à indices.

In [13]:
exo_valeurs = [0,1,0]
exo_dtype = ...  # Remplacer les points par votre réponse.
if exo_dtype is Ellipsis or exo_dtype is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    exo_cibles = torch.tensor(exo_valeurs,dtype=exo_dtype)
    assert exo_cibles.dtype == torch.int64
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

long est un autre nom de int64. Les présences multi-étiquettes demandent un autre contrat.

</details>

### Exercice 2

Trois extraits ont deux descripteurs chacun. Quelle forme reste-t-il en sélectionnant seulement la première ligne ?

In [14]:
exo_x = torch.tensor([[.1,1.5],[.2,4.5],[.15,2.5]])
exo_forme = ...  # Remplacer les points par votre réponse.
if exo_forme is Ellipsis or exo_forme is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert tuple(exo_forme) == tuple(exo_x[0].shape)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

Sélectionner une ligne retire l’axe des extraits ; il reste le vecteur de ses deux mesures.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_valeurs = [0,1,0]
exo_dtype = torch.long
if exo_dtype is Ellipsis or exo_dtype is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    exo_cibles = torch.tensor(exo_valeurs,dtype=exo_dtype)
    assert exo_cibles.dtype == torch.int64
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_x = torch.tensor([[.1,1.5],[.2,4.5],[.15,2.5]])
exo_forme = (2,)
if exo_forme is Ellipsis or exo_forme is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert tuple(exo_forme) == tuple(exo_x[0].shape)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [17]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()